# MitoTox Predictive Modeling Pipeline
## Training a QSAR Model for Mitochondrial Toxicity Prediction

**Input file:** `mito_tox_master_v3.csv` (6,735 compounds, 4 sources)

**What this notebook does — in order:**
1. Load + validate the dataset
2. Drug-likeness filter (remove ionic fragments, solvents)
3. Scaffold-based train/val/test split (the scientifically correct way)
4. Feature engineering (Morgan fingerprints + physicochemical descriptors)
5. Class imbalance handling WITHOUT SMOTE (class weights + threshold tuning)
6. Train 3 models: Random Forest, XGBoost, Logistic Regression
7. 5-fold cross-validation with scaffold grouping
8. Threshold optimization on validation set
9. Final evaluation on held-out test set (MCC, AUC-PR, Recall)
10. SHAP explainability
11. Save final model

**Why NO SMOTE?**
SMOTE synthesizes fake molecular fingerprints by interpolating between real ones.
Interpolated bit vectors do not correspond to any real molecule and have no
biological meaning. For a biological dataset, undersampling the majority class
(removing some safe compounds) is more defensible — you are always working with
real experimental observations.


## Section 1 — Install packages
Run once and wait ~2 minutes.

In [3]:
import numpy as np

print(np.__version__)

1.26.4


In [1]:
# ── CELL 1: Install ──────────────────────────────────────────────────────────
!pip install rdkit xgboost lightgbm shap scikit-learn imbalanced-learn -q

from rdkit import Chem
import xgboost, shap
print("✓ All packages installed")
print(f"  XGBoost: {xgboost.__version__}")
print(f"  SHAP:    {shap.__version__}")


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
contourpy 1.2.0 requires numpy<2.0,>=1.20, but you have numpy 2.0.2 which is incompatible.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.0.2 which is incompatible.


ModuleNotFoundError: No module named 'rdkit'

## Section 2 — Imports

In [ ]:
# ── CELL 2: All imports ──────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings
import pickle
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
warnings.filterwarnings('ignore')

# RDKit
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Descriptors, DataStructs, rdMolDescriptors
from rdkit.Chem.Scaffolds import MurckoScaffold
RDLogger.DisableLog('rdApp.*')

# Sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score, recall_score,
    precision_score, f1_score, matthews_corrcoef,
    precision_recall_curve, roc_curve, confusion_matrix,
    ConfusionMatrixDisplay
)
from sklearn.calibration import CalibratedClassifierCV

# XGBoost
from xgboost import XGBClassifier

# Imbalanced-learn (undersampling only — no SMOTE)
from imblearn.under_sampling import RandomUnderSampler

# SHAP
import shap

print("✓ All imports successful")


## Section 3 — Load, validate, and filter the dataset

### Why we filter for drug-likeness
The Tox21 dataset contains 36 very simple compounds (ethanol, methanol,
hydrazine, ionic fragments like `[Br-]`) that fall completely outside the
chemical space of drug-like molecules. Including them:
- Pollutes the Morgan fingerprint space (0s everywhere for most fragment bits)
- Does not reflect the applicability domain of a mitochondrial toxicity model
- Is standard practice to remove: Tox21 papers routinely apply MW > 100 or
  heavy atom count >= 7 as a minimum drug-like filter.

We apply: **MW >= 100 Da AND heavy atom count >= 7 AND valid RDKit mol object**.
This removes the ionic fragments and solvents while keeping all real drug candidates.


In [ ]:
# ── CELL 3: Load, validate, and drug-likeness filter ─────────────────────────
from google.colab import files
print("Upload mito_tox_master_v3.csv")
uploaded = files.upload()


In [ ]:
# ── CELL 4: Apply filters and compute molecular properties ───────────────────
df_raw = pd.read_csv("mito_tox_master_v3.csv")
print(f"Loaded: {len(df_raw)} rows")
print(f"Before filtering — label counts: {df_raw['label'].value_counts().to_dict()}")
print()

def compute_mol_properties(smiles):
    """Compute MW and heavy atom count. Returns (mol, MW, HAC) or (None,0,0)."""
    try:
        mol = Chem.MolFromSmiles(str(smiles))
        if mol is None:
            return None, 0, 0
        mw  = Descriptors.MolWt(mol)
        hac = mol.GetNumHeavyAtoms()
        return mol, mw, hac
    except Exception:
        return None, 0, 0

rows = []
for _, r in df_raw.iterrows():
    if pd.isna(r['canonical_smiles']):
        continue                          # drop the 1 null SMILES row
    mol, mw, hac = compute_mol_properties(r['canonical_smiles'])
    if mol is None:
        continue                          # drop unparseable SMILES
    if mw < 100 or hac < 7:
        continue                          # drop non-drug-like (solvents, ionic fragments)
    rows.append({
        'canonical_smiles': r['canonical_smiles'],
        'inchikey':         r['inchikey'],
        'label':            int(r['label']),
        'pic50':            r['pic50'],
        'sources':          r['sources'],
        'n_sources':        int(r['n_sources']),
        'mol_wt':           round(mw, 2),
        'heavy_atoms':      hac,
    })

df = pd.DataFrame(rows)
vc = df['label'].value_counts()
print(f"After filtering: {len(df)} compounds "
      f"(removed {len(df_raw) - len(df)} non-drug-like/invalid)")
print(f"  Toxic (1):  {vc[1]:,}  ({100*vc[1]/len(df):.1f}%)")
print(f"  Safe  (0):  {vc[0]:,}  ({100*vc[0]/len(df):.1f}%)")
print(f"  Class ratio: {vc[0]/vc[1]:.2f} : 1  (safe:toxic)")
print()

# Global imbalance ratio — needed for XGBoost
IMBALANCE_RATIO = vc[0] / vc[1]
print(f"Imbalance ratio stored: {IMBALANCE_RATIO:.2f}")


## Section 4 — Scaffold-based train / validation / test split

### Why scaffold splitting (not random splitting)?
Random splitting lets structurally very similar compounds end up in both
the training set and the test set. A model can essentially "memorise" the
activity of a test compound because it trained on a near-identical analogue.
This inflates AUC-ROC by 10-20 percentage points above the real value.

**Bemis-Murcko scaffold splitting** groups compounds by their core scaffold
(rings + linkers, no side chains). All compounds sharing a scaffold are kept
in the SAME split. This means the test set only contains scaffolds the model
has never seen — a much more honest measure of generalisation.

### Split ratios: 70 / 10 / 20
- **Train (70%)**: used for all model fitting and cross-validation
- **Validation (10%)**: used ONLY for threshold tuning after CV (never touches training)
- **Test (20%)**: held out completely until the very end — reported metrics only once

Why 20% test (not 10%)? With ~6,600 compounds, a 20% test set gives ~1,320
compounds. With a 25% toxic prevalence, that's ~330 toxic test compounds —
enough for stable AUC-PR estimates. 10% would give only ~165 toxics, making
PR curve estimates noisy.


In [ ]:
# ── CELL 5: Compute Bemis-Murcko scaffolds ───────────────────────────────────
def get_scaffold(smiles):
    """Return the generic Bemis-Murcko scaffold SMILES for a molecule.
    Compounds with no rings get assigned their canonical SMILES as scaffold
    (each becomes its own singleton group)."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return smiles
    try:
        scaffold = MurckoScaffold.MurckoScaffoldSmiles(
            mol=mol, includeChirality=False
        )
        return scaffold if scaffold else smiles
    except Exception:
        return smiles

print("Computing Bemis-Murcko scaffolds...")
df['scaffold'] = df['canonical_smiles'].apply(get_scaffold)

n_unique_scaffolds = df['scaffold'].nunique()
n_total            = len(df)
singleton_rate     = (df['scaffold'].value_counts() == 1).sum() / n_unique_scaffolds
print(f"  {n_unique_scaffolds:,} unique scaffolds for {n_total:,} compounds")
print(f"  {singleton_rate:.0%} are singleton scaffolds (unique rings or acyclic)")
print(f"  Largest scaffold group: {df['scaffold'].value_counts().iloc[0]} compounds")


In [ ]:
# ── CELL 6: Assign scaffolds to train / val / test ───────────────────────────
np.random.seed(42)

# Sort scaffolds by size descending so we fill splits greedily (largest first)
scaffold_counts = df['scaffold'].value_counts()
scaffolds_sorted = scaffold_counts.index.tolist()

# Target sizes (proportions of total compounds)
target_test  = 0.20
target_val   = 0.10

# Assign each scaffold to a split bucket greedily
scaffold_to_split = {}
test_idx, val_idx, train_idx = [], [], []

# Shuffle scaffolds of the same size randomly before assignment
from itertools import groupby
shuffled_scaffolds = []
for size, group in groupby(scaffolds_sorted, key=lambda s: scaffold_counts[s]):
    group_list = list(group)
    np.random.shuffle(group_list)
    shuffled_scaffolds.extend(group_list)

test_count = val_count = 0

for scaffold in shuffled_scaffolds:
    scaffold_rows = df[df['scaffold'] == scaffold].index.tolist()
    n = len(scaffold_rows)

    if test_count / n_total < target_test:
        test_idx.extend(scaffold_rows)
        test_count += n
    elif val_count / n_total < target_val:
        val_idx.extend(scaffold_rows)
        val_count += n
    else:
        train_idx.extend(scaffold_rows)

df_train = df.loc[train_idx].reset_index(drop=True)
df_val   = df.loc[val_idx  ].reset_index(drop=True)
df_test  = df.loc[test_idx ].reset_index(drop=True)

print("=" * 55)
print("SCAFFOLD SPLIT RESULTS")
print("=" * 55)
for name, split in [("Train (70%)", df_train), ("Val (10%)", df_val), ("Test (20%)", df_test)]:
    vc = split['label'].value_counts()
    pct = len(split)/len(df)*100
    print(f"  {name:15}  {len(split):5} compounds ({pct:.0f}%)")
    print(f"              Toxic={vc.get(1,0)}  Safe={vc.get(0,0)}  "
          f"({100*vc.get(1,0)/len(split):.1f}% toxic)")
print()
print("Unique scaffolds per split (NO OVERLAP = correct):")
train_scaffolds = set(df_train['scaffold'])
val_scaffolds   = set(df_val['scaffold'])
test_scaffolds  = set(df_test['scaffold'])
print(f"  Train: {len(train_scaffolds):,} | Val: {len(val_scaffolds):,} | Test: {len(test_scaffolds):,}")
print(f"  Train∩Test:  {len(train_scaffolds & test_scaffolds)}  (must be 0)")
print(f"  Train∩Val:   {len(train_scaffolds & val_scaffolds)}  (must be 0)")
print(f"  Val∩Test:    {len(val_scaffolds & test_scaffolds)}  (must be 0)")


## Section 5 — Feature engineering

We use three complementary feature types:

| Feature type | Size | What it encodes |
|---|---|---|
| Morgan fingerprint (radius 2) | 2048 bits | Circular substructure environments up to 4 bonds |
| RDKit physicochemical descriptors | 9 values | MW, LogP, TPSA, H-donors, H-acceptors, rotatable bonds, aromatic rings, heteroatoms, Fsp3 |
| Toxicophore flags | 10 bits | Presence of known toxic functional groups (Michael acceptors, nitroaromatics, epoxides, etc.) |

**Total: 2067 features per compound**

Why not just fingerprints alone? The physicochemical descriptors encode
global molecular properties (size, polarity, flexibility) that fingerprints
only capture indirectly. Toxicophore flags directly encode expert knowledge
about which functional groups are known to cause mitochondrial toxicity.


In [ ]:
# ── CELL 7: Define feature functions ─────────────────────────────────────────
MORGAN_RADIUS = 2
MORGAN_NBITS  = 2048

RDKIT_DESC_NAMES = [
    "MolWt", "MolLogP", "NumHDonors", "NumHAcceptors",
    "NumRotatableBonds", "NumAromaticRings", "NumHeteroatoms",
    "FractionCSP3", "TPSA"
]

# Toxicophore SMARTS — known mito-toxic structural alerts
# Sources: Nelms 2015, Gong 2021, Brenk 2008
TOXICOPHORES = {
    "michael_acceptor":         "[CX3]=[CX3][C,S,N]=[O,S]",
    "aldehyde":                 "[CHX3](=O)",
    "epoxide":                  "[OX2r3]1[#6r3][#6r3]1",
    "nitroaromatic":            "c1ccc([N+](=O)[O-])cc1",
    "acyl_halide":              "[CX3](=O)[F,Cl,Br,I]",
    "quinone":                  "O=c1cc[cX3]cc1=O",
    "mito_uncoupler_scaffold":  "Oc1ccc(cc1)[N+](=O)[O-]",
    "thiophene":                "c1ccsc1",
    "aniline":                  "Nc1ccccc1",
    "polycyclic_aromatic":      "c1ccc2cccc3cccc1c23",   # anthracene-like PAH
}
_TOX_PATTERNS = {k: Chem.MolFromSmarts(v) for k, v in TOXICOPHORES.items()}

def featurize(smiles):
    """Convert one SMILES into a feature vector.
    Returns np.array of length 2067 or None if SMILES is invalid."""
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return None

    # 1. Morgan fingerprint (2048 bits)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, MORGAN_RADIUS, MORGAN_NBITS)
    fp_arr = np.zeros(MORGAN_NBITS, dtype=np.float32)
    DataStructs.ConvertToNumpyArray(fp, fp_arr)

    # 2. RDKit physicochemical descriptors (9 values)
    desc = []
    for name in RDKIT_DESC_NAMES:
        if name == "TPSA":
            desc.append(rdMolDescriptors.CalcTPSA(mol))
        else:
            desc.append(getattr(Descriptors, name)(mol))
    desc_arr = np.array(desc, dtype=np.float32)
    desc_arr = np.nan_to_num(desc_arr, nan=0.0)

    # 3. Toxicophore flags (10 bits)
    tox_arr = np.array(
        [1.0 if (pat and mol.HasSubstructMatch(pat)) else 0.0
         for pat in _TOX_PATTERNS.values()],
        dtype=np.float32
    )

    return np.concatenate([fp_arr, desc_arr, tox_arr])


FEATURE_NAMES = (
    [f"fp_{i}" for i in range(MORGAN_NBITS)]
    + RDKIT_DESC_NAMES
    + list(TOXICOPHORES.keys())
)

print(f"Feature vector length: {len(FEATURE_NAMES)}")
print(f"  Morgan FP bits:         {MORGAN_NBITS}")
print(f"  Physicochemical desc:   {len(RDKIT_DESC_NAMES)}")
print(f"  Toxicophore flags:      {len(TOXICOPHORES)}")
print(f"  TOTAL:                  {len(FEATURE_NAMES)}")


In [ ]:
# ── CELL 8: Build feature matrices for all three splits ───────────────────────
def build_matrix(df_split, split_name):
    X_rows, y_rows, valid_idx = [], [], []
    for i, (_, r) in enumerate(df_split.iterrows()):
        feat = featurize(r['canonical_smiles'])
        if feat is not None:
            X_rows.append(feat)
            y_rows.append(r['label'])
            valid_idx.append(i)
    X = np.array(X_rows, dtype=np.float32)
    y = np.array(y_rows, dtype=np.int32)
    print(f"  {split_name}: {X.shape[0]} compounds × {X.shape[1]} features "
          f"| Toxic={y.sum()} Safe={(y==0).sum()}")
    return X, y

print("Building feature matrices...")
X_train, y_train = build_matrix(df_train, "Train")
X_val,   y_val   = build_matrix(df_val,   "Val  ")
X_test,  y_test  = build_matrix(df_test,  "Test ")
print()
print(f"✓ Ready for modeling")


## Section 6 — Class imbalance handling (no SMOTE)

### The three-layer strategy

**Layer 1 — class_weight (all models)**
Tell the model that misclassifying a toxic compound costs more than
misclassifying a safe one:
- `w_toxic = n_total / (2 × n_toxic)` = ~1.96
- `w_safe  = n_total / (2 × n_safe)`  = ~0.67
- Misclassifying one toxic compound costs ~2.9× more than one safe compound

**Layer 2 — scale_pos_weight (XGBoost)**
XGBoost uses a different parameter: `scale_pos_weight = n_negative / n_positive`
(= the safe:toxic ratio ≈ 2.9). This multiplies the gradient of toxic examples.

**Layer 3 — Threshold tuning on the validation set**
Default threshold = 0.5 means "predict toxic if P(toxic) > 50%."
With an imbalanced dataset, a threshold of 0.5 is almost always too high —
the model is conservative about predicting the minority class.
We sweep thresholds from 0.05 to 0.95 and pick the one maximising MCC
on the validation set. This is the Tang et al. 2020 "threshold-moving" method.

**Why NOT SMOTE?** Every SMOTE synthetic sample is a linear interpolation
between two real fingerprint bit vectors: `x_new = x_i + λ(x_j - x_i)`.
The resulting bit vector does not correspond to any real molecule and has
no chemical or biological meaning. It will also fail the Lipinski/drug-likeness
filter we just applied. We keep only real experimental data.


In [ ]:
# ── CELL 9: Compute class weights ────────────────────────────────────────────
from sklearn.utils.class_weight import compute_class_weight

classes = np.array([0, 1])
weights = compute_class_weight('balanced', classes=classes, y=y_train)
CLASS_WEIGHT_DICT = {0: weights[0], 1: weights[1]}

print("Class weights (balanced formula: n / (n_classes × count_j)):")
print(f"  Safe  (0): {CLASS_WEIGHT_DICT[0]:.4f}")
print(f"  Toxic (1): {CLASS_WEIGHT_DICT[1]:.4f}")
print(f"  Ratio (how much more toxic counts): "
      f"{CLASS_WEIGHT_DICT[1]/CLASS_WEIGHT_DICT[0]:.2f}×")
print()
print(f"XGBoost scale_pos_weight = {IMBALANCE_RATIO:.2f}")
print("(This is n_safe/n_toxic — XGBoost's equivalent of class weights)")


## Section 7 — Evaluation metric suite

### Why these metrics (and not accuracy)

If 74% of our dataset is safe, a model predicting "always safe" gets 74%
accuracy — but 0% recall. Accuracy is useless here.

| Metric | Formula | Why it matters |
|---|---|---|
| **AUC-PR** | Area under Precision-Recall curve | Best single metric for imbalanced data; baseline = prevalence (~0.26) |
| **MCC** | (TP·TN - FP·FN) / √((TP+FP)(TP+FN)(TN+FP)(TN+FN)) | Uses all 4 confusion matrix cells; +1=perfect, 0=random, -1=inverse |
| **Recall (Sensitivity)** | TP / (TP+FN) | What fraction of actual toxics did we catch? Target: ≥ 0.75 |
| **AUC-ROC** | Area under ROC curve | Standard but optimistic for imbalanced; included for benchmark comparison |
| **F1** | 2·P·R / (P+R) | Harmonic mean of precision and recall |

**Primary metric: AUC-PR. Secondary: MCC.**


In [ ]:
# ── CELL 10: Evaluation function ─────────────────────────────────────────────
def evaluate(model, X, y, threshold=0.5, prefix=""):
    """Compute and print the full metric suite for one model/split combination."""
    y_proba = model.predict_proba(X)[:, 1]
    y_pred  = (y_proba >= threshold).astype(int)

    metrics = {
        'AUC-ROC':   roc_auc_score(y, y_proba),
        'AUC-PR':    average_precision_score(y, y_proba),
        'Recall':    recall_score(y, y_pred, zero_division=0),
        'Precision': precision_score(y, y_pred, zero_division=0),
        'F1':        f1_score(y, y_pred, zero_division=0),
        'MCC':       matthews_corrcoef(y, y_pred),
    }
    if prefix:
        print(f"  [{prefix}]")
    for k, v in metrics.items():
        marker = " ← PRIMARY" if k == 'AUC-PR' else (" ← SECONDARY" if k == 'MCC' else "")
        print(f"    {k:<12} {v:.4f}{marker}")
    return metrics, y_proba

def find_best_threshold(y_true, y_proba, metric='mcc'):
    """
    Sweep thresholds 0.05–0.95 and return the one maximising MCC (or F1).
    Applied to the VALIDATION set only — never the test set.
    """
    best_thresh = 0.5
    best_score  = -999
    thresholds  = np.arange(0.05, 0.95, 0.01)
    for t in thresholds:
        y_pred = (y_proba >= t).astype(int)
        if metric == 'mcc':
            score = matthews_corrcoef(y_true, y_pred)
        else:
            score = f1_score(y_true, y_pred, zero_division=0)
        if score > best_score:
            best_score  = score
            best_thresh = t
    return best_thresh, best_score

print("✓ Evaluation functions defined")


## Section 8 — 5-fold Scaffold-Aware Cross-Validation

### Why 5 folds (not 10)?
With ~4,600 training compounds, 5-fold gives ~920 per validation fold.
The toxic class has ~1,180 compounds in training — 5-fold leaves ~236 toxics
per fold, which is sufficient for stable AUC-PR estimates.
10-fold would give only ~118 toxics per validation fold, making per-fold
metrics noisier without meaningful additional information.

### Scaffold-aware grouping in CV
We use `GroupKFold` with scaffold as the group key. This ensures that
within cross-validation, no scaffold appears in both the CV-train and
CV-val folds — the same guarantee we applied at the outer train/test split.
This is the setup that gave AUC-ROC 0.66–0.88 (honest) vs 0.90–0.95
(inflated random) in your previous notebook.


In [ ]:
# ── CELL 11: Define models ────────────────────────────────────────────────────
# Three models covering different algorithmic families

# Model 1: Random Forest
# - Ensemble of decision trees; naturally handles mixed feature types
# - class_weight handles imbalance at the split level (each split weighted)
# - max_features='sqrt' prevents any single descriptor dominating
rf_model = RandomForestClassifier(
    n_estimators    = 500,
    max_features    = 'sqrt',
    min_samples_leaf= 2,
    class_weight    = CLASS_WEIGHT_DICT,
    random_state    = 42,
    n_jobs          = -1,
)

# Model 2: XGBoost
# - Gradient boosted trees; often best performance on tabular data
# - scale_pos_weight = n_safe/n_toxic (XGBoost's class weight mechanism)
# - eval_metric='aucpr' trains directly on AUC-PR (our primary metric)
xgb_model = XGBClassifier(
    n_estimators    = 500,
    max_depth       = 6,
    learning_rate   = 0.05,
    subsample       = 0.8,
    colsample_bytree= 0.8,
    scale_pos_weight= IMBALANCE_RATIO,   # n_safe / n_toxic
    eval_metric     = 'aucpr',
    use_label_encoder=False,
    random_state    = 42,
    n_jobs          = -1,
    verbosity       = 0,
)

# Model 3: Logistic Regression (interpretable baseline)
# - class_weight='balanced' adjusts sample weights in the loss function
# - C=0.1 (moderate L2 regularisation — important with 2067 features)
lr_model = LogisticRegression(
    C            = 0.1,
    class_weight = CLASS_WEIGHT_DICT,
    max_iter     = 1000,
    solver       = 'lbfgs',
    random_state = 42,
    n_jobs       = -1,
)

MODELS = {
    'RandomForest':      rf_model,
    'XGBoost':           xgb_model,
    'LogisticRegression':lr_model,
}
print("✓ 3 models defined:")
for name in MODELS:
    print(f"   {name}")


In [ ]:
# ── CELL 12: 5-fold scaffold-aware CV ────────────────────────────────────────
# Group by scaffold so the same scaffold never appears in both
# the CV-train and CV-val fold simultaneously.

from sklearn.model_selection import GroupKFold

# Assign integer group IDs from scaffold strings
scaffold_ids = df_train['scaffold'].astype('category').cat.codes.values
gkf = GroupKFold(n_splits=5)

all_cv_results = {}

for model_name, model in MODELS.items():
    print(f"\n{'='*55}")
    print(f"  CV: {model_name}")
    print(f"{'='*55}")
    fold_metrics = []

    for fold, (tr_idx, vl_idx) in enumerate(
            gkf.split(X_train, y_train, groups=scaffold_ids), 1):
        X_tr, X_vl = X_train[tr_idx], X_train[vl_idx]
        y_tr, y_vl = y_train[tr_idx], y_train[vl_idx]

        model.fit(X_tr, y_tr)

        y_proba_vl = model.predict_proba(X_vl)[:, 1]
        auc_pr  = average_precision_score(y_vl, y_proba_vl)
        auc_roc = roc_auc_score(y_vl, y_proba_vl)
        # Use default threshold 0.5 during CV
        y_pred  = (y_proba_vl >= 0.5).astype(int)
        mcc     = matthews_corrcoef(y_vl, y_pred)
        recall  = recall_score(y_vl, y_pred, zero_division=0)

        fold_metrics.append({
            'AUC-PR': auc_pr, 'AUC-ROC': auc_roc,
            'MCC': mcc, 'Recall': recall
        })
        print(f"  Fold {fold}:  AUC-PR={auc_pr:.4f}  AUC-ROC={auc_roc:.4f}  "
              f"MCC={mcc:.4f}  Recall={recall:.4f}")

    cv_df = pd.DataFrame(fold_metrics)
    print(f"  ── Mean ± Std ──")
    for col in cv_df.columns:
        print(f"  {col:<12} {cv_df[col].mean():.4f} ± {cv_df[col].std():.4f}")
    all_cv_results[model_name] = cv_df

print("\n✓ Cross-validation complete")


## Section 9 — Final training on full train set + threshold tuning

Now that CV tells us each model's expected performance, we retrain each model
on the **complete training set** (all 5 folds combined) and tune the
classification threshold using the **validation set**.

The threshold tuning sweeps from 0.05 to 0.95 in steps of 0.01 and picks
the threshold that maximises MCC on the validation set.
This threshold is then applied when evaluating the final test set.


In [ ]:
# ── CELL 13: Retrain on full train set + tune threshold on val set ───────────
trained_models   = {}
best_thresholds  = {}
val_proba_cache  = {}

for model_name, model in MODELS.items():
    print(f"\nTraining {model_name} on full training set...")
    model.fit(X_train, y_train)
    trained_models[model_name] = model

    # Tune threshold on validation set
    y_val_proba = model.predict_proba(X_val)[:, 1]
    val_proba_cache[model_name] = y_val_proba

    best_t, best_mcc = find_best_threshold(y_val, y_val_proba, metric='mcc')
    best_thresholds[model_name] = best_t

    print(f"  Best threshold (maximises MCC on val): {best_t:.2f}  "
          f"(Val MCC at this threshold: {best_mcc:.4f})")
    print(f"  Default 0.5 MCC for comparison: "
          f"{matthews_corrcoef(y_val, (y_val_proba>=0.5).astype(int)):.4f}")

print("\n✓ All models trained. Thresholds locked.")
print("\nFinal thresholds to use on test set:")
for name, t in best_thresholds.items():
    print(f"  {name}: {t:.2f}")


## Section 10 — Final evaluation on the held-out test set

This is the ONLY time we look at the test set. We apply the threshold
tuned on the validation set and report all metrics.


In [ ]:
# ── CELL 14: Test set evaluation ─────────────────────────────────────────────
print("=" * 55)
print("  FINAL TEST SET RESULTS")
print("=" * 55)

test_results = {}
for model_name, model in trained_models.items():
    t = best_thresholds[model_name]
    print(f"\n  {model_name}  (threshold={t:.2f})")
    metrics, y_test_proba = evaluate(model, X_test, y_test, threshold=t)
    test_results[model_name] = {'metrics': metrics, 'proba': y_test_proba}

# Summary comparison table
print("\n" + "=" * 55)
print("  BENCHMARK COMPARISON TABLE")
print("=" * 55)
summary = []
for name, res in test_results.items():
    m = res['metrics']
    summary.append({
        'Model': name,
        'AUC-PR': f"{m['AUC-PR']:.4f}",
        'AUC-ROC':f"{m['AUC-ROC']:.4f}",
        'MCC':    f"{m['MCC']:.4f}",
        'Recall': f"{m['Recall']:.4f}",
        'F1':     f"{m['F1']:.4f}",
    })
print(pd.DataFrame(summary).to_string(index=False))
print()
print("Literature benchmarks for reference:")
print("  Hemmerich 2020:  AUC-ROC ~0.80–0.84  (random split)")
print("  XML-CIMT 2022:   Accuracy ~0.87       (random split, not directly comparable)")
print("  Your scaffold-split AUC-ROC is expected to be lower than random-split")
print("  literature values — this is correct and more honest.")


In [ ]:
# ── CELL 15: Confusion matrices and ROC/PR curves ────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle('Model Evaluation — Test Set', fontsize=14, fontweight='bold')

model_names = list(trained_models.keys())
colors = ['steelblue', 'darkorange', 'forestgreen']

for col, (name, color) in enumerate(zip(model_names, colors)):
    model  = trained_models[name]
    t      = best_thresholds[name]
    proba  = test_results[name]['proba']
    y_pred = (proba >= t).astype(int)

    # Top row: Confusion matrix
    ax = axes[0, col]
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Safe', 'Toxic'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{name}\n(threshold={t:.2f})', fontsize=9)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

    # Bottom row: PR curve
    ax2 = axes[1, col]
    prec, rec, _ = precision_recall_curve(y_test, proba)
    auc_pr = average_precision_score(y_test, proba)
    prevalence = y_test.mean()
    ax2.plot(rec, prec, color=color, lw=2, label=f'AUC-PR={auc_pr:.3f}')
    ax2.axhline(prevalence, color='gray', ls='--', lw=1,
                label=f'Baseline={prevalence:.2f}')
    ax2.set_xlabel('Recall')
    ax2.set_ylabel('Precision')
    ax2.set_title(f'Precision-Recall Curve\n{name}', fontsize=9)
    ax2.legend(fontsize=8)
    ax2.set_xlim([0, 1])
    ax2.set_ylim([0, 1])

plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved → model_evaluation.png")


## Section 11 — SHAP Explainability

SHAP (SHapley Additive exPlanations) tells us **which features pushed each
prediction toward Toxic or Safe**, computed as the average marginal contribution
of each feature across all possible orderings.

We use `TreeExplainer` (fast, exact for tree-based models).
The beeswarm plot shows the top 25 most important features — each point is
one compound, colored by feature value (red=high, blue=low), and positioned
by the SHAP value (how much it pushed the prediction toward Toxic on the x-axis).


In [ ]:
# ── CELL 16: SHAP on Random Forest (best for fingerprint data) ───────────────
print("Computing SHAP values for Random Forest...")
print("(Takes 2-3 minutes on the test set)")

rf = trained_models['RandomForest']
explainer   = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)

# shap_values is a list [safe_shap, toxic_shap]
# We want toxic_shap (index 1)
sv_toxic = shap_values[1]

# Summary beeswarm plot — top 25 features
plt.figure(figsize=(10, 9))
shap.summary_plot(
    sv_toxic, X_test,
    feature_names=FEATURE_NAMES,
    max_display=25,
    show=False,
    plot_type='dot',
)
plt.title('SHAP Feature Importance — Random Forest (Toxic class)', fontsize=12)
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved → shap_summary.png")


In [ ]:
# ── CELL 17: Top toxicophore analysis ────────────────────────────────────────
# Extract which toxicophore flags had the highest mean |SHAP| value
# This tells you which structural alerts your model relies on most

# Toxicophore feature indices
tox_start = MORGAN_NBITS + len(RDKIT_DESC_NAMES)
tox_names  = list(TOXICOPHORES.keys())

mean_abs_shap = np.abs(sv_toxic).mean(axis=0)
tox_shap = [(tox_names[i], mean_abs_shap[tox_start + i])
            for i in range(len(tox_names))]
tox_shap.sort(key=lambda x: -x[1])

print("Toxicophore flag importance (mean |SHAP|):")
for name, val in tox_shap:
    bar = '█' * int(val * 500)
    print(f"  {name:<30} {val:.5f}  {bar}")

print()
print("Physicochemical descriptor importance:")
desc_shap = [(RDKIT_DESC_NAMES[i], mean_abs_shap[MORGAN_NBITS + i])
             for i in range(len(RDKIT_DESC_NAMES))]
desc_shap.sort(key=lambda x: -x[1])
for name, val in desc_shap:
    print(f"  {name:<25} {val:.5f}")


## Section 12 — Save models and results

In [ ]:
# ── CELL 18: Save everything ─────────────────────────────────────────────────
import pickle

# Save all three trained models
for name, model in trained_models.items():
    filename = f"mitotox_{name.lower()}_model.pkl"
    with open(filename, 'wb') as f:
        pickle.dump(model, f)
    print(f"✓ Saved {filename}")

# Save thresholds
thresholds_df = pd.DataFrame([
    {'model': k, 'threshold': v} for k, v in best_thresholds.items()
])
thresholds_df.to_csv('model_thresholds.csv', index=False)
print("✓ Saved model_thresholds.csv")

# Save test set results
results_rows = []
for name, res in test_results.items():
    row = {'model': name}
    row.update(res['metrics'])
    results_rows.append(row)
pd.DataFrame(results_rows).to_csv('test_set_results.csv', index=False)
print("✓ Saved test_set_results.csv")

# Save CV results
for name, cv_df in all_cv_results.items():
    cv_df.to_csv(f'cv_results_{name}.csv', index=False)
print("✓ Saved CV result CSVs")

# Download all outputs
from google.colab import files
for f in ['test_set_results.csv', 'model_thresholds.csv',
          'model_evaluation.png', 'shap_summary.png']:
    try:
        files.download(f)
    except Exception:
        pass

print()
print("=" * 55)
print("  PIPELINE COMPLETE")
print("=" * 55)
print()
print("  Files saved:")
print("  - mitotox_randomforest_model.pkl")
print("  - mitotox_xgboost_model.pkl")
print("  - mitotox_logisticregression_model.pkl")
print("  - model_thresholds.csv")
print("  - test_set_results.csv")
print("  - model_evaluation.png")
print("  - shap_summary.png")
